# State Bench Long-Context: Transformer vs Hybrid vs GDN

This notebook compares matched 275M transformer, hybrid, and GDN checkpoints on the `jacksonp-state-bench-lc` tasks. It reads per-example prediction artifacts, derives exact prompt-token length as `num_tokens_all - num_tokens`, and supports task-level and accuracy-versus-prompt-length analysis.

In [ ]:
from __future__ import annotations

import json
import math
import re
import sys
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "aesthetics.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
FIGURES_DIR = PROJECT_ROOT / "figures" / "sb_lc"
import aesthetics as aes  # noqa: E402

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)

## Configuration

In [ ]:
RESULTS_PATH = PROJECT_ROOT / "results" / "jacksonp-state-bench-lc" / "all_results.json"
TASK_RE = re.compile(r"state_bench_lc_(.+)_(aperiodic|periodic|r_trivial)$")
COMPLEXITY_ORDER = ["aperiodic", "periodic", "r_trivial"]
MODEL_ORDER = ["Transformer", "Hybrid", "GDN"]
REFERENCE_MODEL = "Transformer"
COMPARISON_MODELS = ["Hybrid", "GDN"]
MODEL_COLORS = {"Transformer": "#DD0426", "Hybrid": "#8304DE", "GDN": "#046ADE"}
# Even-width log10 bins are appropriate for prompts spanning 17 to ~131k tokens.
LENGTH_EDGES = np.logspace(1, np.log10(200_000), 16)
MIN_BIN_EXAMPLES = 30


def model_label(result: dict) -> str:
    text = f"{result.get('experiment_name', '')} {result.get('model', '')}".lower()
    if "gdn" in text:
        return "GDN"
    if "transformer" in text:
        return "Transformer"
    if "hybrid" in text:
        return "Hybrid"
    raise ValueError(f"Unrecognized architecture: {text}")


def nested_metric(metrics: dict, metric: str) -> float:
    value = metrics
    for key in metric.split(":"):
        value = value[key]
    return float(value)


def artifact_path(result: dict, task: dict, kind: str) -> Path:
    root = RESULTS_PATH.parent / result["experiment_id"] / "main" / kind
    matches = sorted(root.glob(f"*/*_{task['task_hash'][-6:]}-{kind}.jsonl"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {kind} artifact for {task['task']}; found {matches}"
        )
    return matches[0]

## Load per-example results

The three runs evaluate the same examples. Prompt tokens come directly from each model output; all answer choices for an example share the same derived prompt length.

In [ ]:
raw_results = json.loads(RESULTS_PATH.read_text())
rows = []
for result in raw_results:
    model = model_label(result)
    for task in result["tasks"]:
        match = TASK_RE.fullmatch(task["task"])
        if match is None:
            continue
        domain, complexity = match.groups()
        metric = task["primary_metric"]
        request_stream = (
            artifact_path(result, task, "requests").open()
            if complexity == "r_trivial"
            else None
        )
        with artifact_path(result, task, "predictions").open() as stream:
            for line in stream:
                prediction = json.loads(line)
                request = (
                    json.loads(next(request_stream))
                    if request_stream is not None
                    else {}
                )
                doc = request.get("doc", {})
                if request and prediction["doc_id"] != request["doc_id"]:
                    raise ValueError(
                        f"Prediction/request mismatch: {task['task']} doc {prediction['doc_id']}"
                    )
                prompt_lengths = [
                    output["num_tokens_all"] - output["num_tokens"]
                    for output in prediction["model_output"]
                ]
                if len(set(prompt_lengths)) != 1:
                    raise ValueError(
                        f"Choice-dependent prompt length: {task['task']} doc {prediction['doc_id']}"
                    )
                rows.append(
                    {
                        "model": model,
                        "task": task["task"],
                        "domain": domain.replace("_", " ").title(),
                        "complexity": complexity.replace("_", "-"),
                        "doc_id": prediction["doc_id"],
                        "native_id": prediction["native_id"],
                        "prompt_tokens": prompt_lengths[0],
                        "score": nested_metric(prediction["instance_metrics"], metric),
                        "padding_lines": doc.get("num_filler_lines"),
                        "padding_complexity": doc.get("filler_distraction")
                        or ("none" if doc else None),
                    }
                )
        if request_stream is not None:
            request_stream.close()

item_df = pd.DataFrame(rows)
item_df["model"] = pd.Categorical(item_df["model"], MODEL_ORDER, ordered=True)
item_df["complexity"] = pd.Categorical(
    item_df["complexity"],
    [value.replace("_", "-") for value in COMPLEXITY_ORDER],
    ordered=True,
)
print(
    f"Loaded {len(item_df):,} example/model rows across {item_df['task'].nunique()} task variants"
)
display(
    item_df.groupby("model", observed=True).agg(
        examples=("score", "size"),
        accuracy=("score", "mean"),
        min_tokens=("prompt_tokens", "min"),
        max_tokens=("prompt_tokens", "max"),
    )
)

## Task-level performance

In [ ]:
task_df = item_df.groupby(
    ["domain", "complexity", "model"], observed=True, as_index=False
).agg(accuracy=("score", "mean"), n=("score", "size"))
task_table = task_df.pivot(
    index=["domain", "complexity"], columns="model", values="accuracy"
)
task_delta_columns = []
for model in COMPARISON_MODELS:
    column = f"{model} - {REFERENCE_MODEL}"
    task_table[column] = task_table[model] - task_table[REFERENCE_MODEL]
    task_delta_columns.append(column)
display(
    task_table.style.format("{:.1%}").background_gradient(
        subset=task_delta_columns, cmap="PiYG", vmin=-0.12, vmax=0.12
    )
)

plot_task_df = task_df.copy()
plot_task_df["task label"] = (
    plot_task_df["domain"] + " · " + plot_task_df["complexity"].astype(str)
)
fig, ax = plt.subplots(figsize=(aes.PAPER_WIDTH_IN, 4.2))
sns.barplot(
    data=plot_task_df,
    y="task label",
    x="accuracy",
    hue="model",
    hue_order=MODEL_ORDER,
    palette=MODEL_COLORS,
    ax=ax,
)
ax.set(
    xlabel="Accuracy", ylabel=None, xlim=(0, 1), title="State Bench LC accuracy by task"
)
ax.xaxis.set_major_formatter(aes.PCT_FORMATTER)
ax.legend(title=None, frameon=False, loc="lower right")
sns.despine()
fig.tight_layout()
_ = aes.save_figure(FIGURES_DIR / "task_accuracy_by_task", fig=fig, save_png=True)
plt.show()

## Accuracy vs prompt length

Points use powers-of-two prompt-token bins; bands are Wilson 95% intervals. Edit `LENGTH_EDGES` or `MIN_BIN_EXAMPLES` above to change the resolution.

In [ ]:
def wilson_interval(successes: pd.Series, n: pd.Series) -> tuple[pd.Series, pd.Series]:
    z = 1.96
    p = successes / n
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    radius = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denominator
    return center - radius, center + radius


item_df["length_bin"] = pd.cut(
    item_df["prompt_tokens"], LENGTH_EDGES, right=False, include_lowest=True
)
item_df["length_mid"] = (
    item_df["length_bin"]
    .map(
        lambda value: math.sqrt(value.left * value.right) if pd.notna(value) else np.nan
    )
    .astype(float)
)
length_df = item_df.groupby(
    ["domain", "complexity", "model", "length_bin", "length_mid"],
    observed=True,
    as_index=False,
).agg(accuracy=("score", "mean"), successes=("score", "sum"), n=("score", "size"))
length_df = length_df[length_df["n"] >= MIN_BIN_EXAMPLES].copy()
length_df["ci_low"], length_df["ci_high"] = wilson_interval(
    length_df["successes"], length_df["n"]
)

FORMAT_ORDER = [
    "Cube Painting",
    "Integer Code",
    "People In Rooms",
    "Spreadsheet Cells",
    "Status Lights",
]
MONOID_COMPLEXITY_ORDER = ["r-trivial", "aperiodic", "periodic"]
fig, axes = plt.subplots(
    len(FORMAT_ORDER),
    len(MONOID_COMPLEXITY_ORDER),
    figsize=(aes.PAPER_WIDTH_IN * 1.45, 7),
    sharex=True,
    sharey=True,
)
for row_idx, formatting in enumerate(FORMAT_ORDER):
    for col_idx, complexity in enumerate(MONOID_COMPLEXITY_ORDER):
        ax = axes[row_idx, col_idx]
        subset = length_df[
            (length_df["domain"] == formatting)
            & (length_df["complexity"] == complexity)
        ]
        for model in MODEL_ORDER:
            curve = subset[subset["model"] == model].sort_values("length_mid")
            x = curve["length_mid"].to_numpy(float)
            y = curve["accuracy"].to_numpy(float)
            ax.plot(
                x, y, marker="o", ms=2.5, lw=1.2, color=MODEL_COLORS[model], label=model
            )
            ax.fill_between(
                x,
                curve["ci_low"].to_numpy(float),
                curve["ci_high"].to_numpy(float),
                color=MODEL_COLORS[model],
                alpha=0.12,
            )
        ax.set_xscale("log", base=10)
        ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.2)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        if row_idx == 0:
            ax.set_title(complexity.title(), fontsize=9)
        if col_idx == 0:
            ax.set_ylabel(formatting, fontsize=9)
fig.supxlabel("Prompt tokens (log scale)")
handles = [
    plt.Line2D([0], [0], color=MODEL_COLORS[m], marker="o", label=m)
    for m in MODEL_ORDER
]
fig.legend(
    handles=handles,
    loc="upper center",
    frameon=False,
    ncols=len(MODEL_ORDER),
    bbox_to_anchor=(0.5, 1.04),
)
# fig.suptitle("Accuracy vs prompt length by task", y=1.01)
fig.tight_layout(rect=(0, 0, 0.93, 1))
_ = aes.save_figure(FIGURES_DIR / "accuracy_by_prompt_length", fig=fig, save_png=True)
plt.show()

### Summary across formatting

This view pools examples across formatting within each monoid complexity. Because r-trivial examples only use Integer Code formatting, its panel contains that formatting alone.

In [ ]:
summary_length_df = (
    item_df.groupby(
        ["complexity", "model", "length_bin", "length_mid"],
        observed=True,
        as_index=False,
    )
    .agg(
        accuracy=("score", "mean"),
        successes=("score", "sum"),
        n=("score", "size"),
    )
    .query("n >= @MIN_BIN_EXAMPLES")
    .copy()
)
summary_length_df["ci_low"], summary_length_df["ci_high"] = wilson_interval(
    summary_length_df["successes"], summary_length_df["n"]
)

fig, axes = plt.subplots(
    1,
    len(MONOID_COMPLEXITY_ORDER),
    figsize=(aes.PAPER_WIDTH_IN * 1.45, 2.6),
    sharex=True,
    sharey=True,
)
for ax, complexity in zip(axes, MONOID_COMPLEXITY_ORDER, strict=True):
    subset = summary_length_df[summary_length_df["complexity"] == complexity]
    for model in MODEL_ORDER:
        curve = subset[subset["model"] == model].sort_values("length_mid")
        x = curve["length_mid"].to_numpy(float)
        y = curve["accuracy"].to_numpy(float)
        ax.plot(
            x, y, marker="o", ms=2.5, lw=1.2, color=MODEL_COLORS[model], label=model
        )
        ax.fill_between(
            x,
            curve["ci_low"].to_numpy(float),
            curve["ci_high"].to_numpy(float),
            color=MODEL_COLORS[model],
            alpha=0.12,
        )
    ax.set_xscale("log", base=10)
    ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.2)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.set_title(complexity.title(), fontsize=9)
axes[0].set_ylabel("Accuracy")
fig.supxlabel("Prompt tokens (log scale)")
handles = [
    plt.Line2D([0], [0], color=MODEL_COLORS[m], marker="o", label=m)
    for m in MODEL_ORDER
]
fig.legend(
    handles=handles,
    loc="upper center",
    frameon=False,
    ncols=len(MODEL_ORDER),
    bbox_to_anchor=(0.5, 1.05),
)
fig.tight_layout(rect=(0, 0, 1, 0.92))
_ = aes.save_figure(
    FIGURES_DIR / "accuracy_by_prompt_length_aggregated",
    fig=fig,
    save_png=True,
)
plt.show()

## R-trivial breakout A: padding amount

For r-trivial Integer Code examples, padding amount is the dataset's `num_filler_lines`: 0, 18, 198, or 1,998 filler lines. The table and context-length plot marginalize over padding complexity within each nonzero amount.

In [ ]:
rtrivial_df = item_df[item_df["complexity"] == "r-trivial"].copy()
rtrivial_df["padding_lines"] = rtrivial_df["padding_lines"].astype(int)
padding_amount_df = rtrivial_df.groupby(
    ["padding_lines", "model"], observed=True, as_index=False
).agg(
    accuracy=("score", "mean"),
    n=("score", "size"),
    median_prompt_tokens=("prompt_tokens", "median"),
)
padding_amount_table = padding_amount_df.pivot(
    index="padding_lines", columns="model", values="accuracy"
)
padding_amount_delta_columns = []
for model in COMPARISON_MODELS:
    column = f"{model} - {REFERENCE_MODEL}"
    padding_amount_table[column] = (
        padding_amount_table[model] - padding_amount_table[REFERENCE_MODEL]
    )
    padding_amount_delta_columns.append(column)
display(
    padding_amount_table.style.format("{:.1%}").background_gradient(
        subset=padding_amount_delta_columns, cmap="PiYG", vmin=-0.15, vmax=0.15
    )
)
display(
    padding_amount_df.pivot(
        index="padding_lines", columns="model", values=["n", "median_prompt_tokens"]
    )
)

fig, ax = plt.subplots(figsize=(aes.PAPER_WIDTH_IN * 0.75, 2.8))
for model in MODEL_ORDER:
    curve = padding_amount_df[padding_amount_df["model"] == model].sort_values(
        "padding_lines"
    )
    ax.plot(
        curve["median_prompt_tokens"],
        curve["accuracy"],
        marker="o",
        lw=1.5,
        color=MODEL_COLORS[model],
        label=model,
    )
ax.set(
    xlabel="Context length (prompt tokens)",
    ylabel="Accuracy",
    ylim=(0, 1),
    title="R-trivial accuracy vs context length, padding amounts pooled",
)
ax.set_xscale("log", base=10)
ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, title=None)
sns.despine()
fig.tight_layout()
_ = aes.save_figure(FIGURES_DIR / "r_trivial_padding_amount", fig=fig, save_png=True)
plt.show()

## R-trivial breakout B: padding complexity

Padding complexity is the dataset's `filler_distraction` level (`easy`, `medium`, or `hard`). The table marginalizes over padding amount. The context-length panels retain each padding amount separately and repeat the unpadded control as the first point in each panel.

In [ ]:
PADDING_COMPLEXITY_ORDER = ["none", "easy", "medium", "hard"]
rtrivial_df["padding_complexity"] = pd.Categorical(
    rtrivial_df["padding_complexity"], PADDING_COMPLEXITY_ORDER, ordered=True
)
padding_complexity_df = rtrivial_df.groupby(
    ["padding_complexity", "model"], observed=True, as_index=False
).agg(
    accuracy=("score", "mean"),
    n=("score", "size"),
    median_prompt_tokens=("prompt_tokens", "median"),
)
padding_complexity_table = padding_complexity_df.pivot(
    index="padding_complexity", columns="model", values="accuracy"
)
padding_complexity_delta_columns = []
for model in COMPARISON_MODELS:
    column = f"{model} - {REFERENCE_MODEL}"
    padding_complexity_table[column] = (
        padding_complexity_table[model] - padding_complexity_table[REFERENCE_MODEL]
    )
    padding_complexity_delta_columns.append(column)
display(
    padding_complexity_table.style.format("{:.1%}").background_gradient(
        subset=padding_complexity_delta_columns, cmap="PiYG", vmin=-0.15, vmax=0.15
    )
)
display(
    padding_complexity_df.pivot(
        index="padding_complexity",
        columns="model",
        values=["n", "median_prompt_tokens"],
    )
)

padding_complexity_length_df = rtrivial_df.groupby(
    ["padding_complexity", "padding_lines", "model"],
    observed=True,
    as_index=False,
).agg(
    accuracy=("score", "mean"),
    n=("score", "size"),
    median_prompt_tokens=("prompt_tokens", "median"),
)
control_df = padding_complexity_length_df[
    padding_complexity_length_df["padding_complexity"] == "none"
]
plotted_complexities = ["easy", "medium", "hard"]
fig, axes = plt.subplots(
    1, 3, figsize=(aes.PAPER_WIDTH_IN * 1.35, 2.8), sharex=True, sharey=True
)
for ax, complexity in zip(axes, plotted_complexities, strict=True):
    complexity_df = padding_complexity_length_df[
        padding_complexity_length_df["padding_complexity"] == complexity
    ]
    for model in MODEL_ORDER:
        curve = pd.concat(
            [
                control_df[control_df["model"] == model],
                complexity_df[complexity_df["model"] == model],
            ],
            ignore_index=True,
        ).sort_values("median_prompt_tokens")
        ax.plot(
            curve["median_prompt_tokens"],
            curve["accuracy"],
            marker="o",
            lw=1.5,
            color=MODEL_COLORS[model],
            label=model,
        )
    ax.set_xscale("log", base=10)
    ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
    ax.set_title(complexity.title())
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Accuracy")
handles = [
    plt.Line2D([0], [0], color=MODEL_COLORS[model], marker="o", label=model)
    for model in MODEL_ORDER
]
fig.supxlabel("Context length (prompt tokens)")
fig.legend(handles=handles, loc="center right", frameon=False)
fig.suptitle("R-trivial accuracy vs context length by padding complexity", y=1.02)
sns.despine()
fig.tight_layout(rect=(0, 0, 0.91, 1))
_ = aes.save_figure(
    FIGURES_DIR / "r_trivial_padding_complexity", fig=fig, save_png=True
)
plt.show()

## Log-length trend models

For each task, a binomial GLM estimates `accuracy ~ model * centered log10(prompt tokens)`, using Transformer as the reference. Each architecture coefficient is its log-odds advantage over Transformer at that task's median length; the interaction is the change in that advantage for each 10× increase in prompt length. These linear log-length trends summarize the richer curves above.

In [ ]:
trend_rows = []
for (domain, complexity), group in item_df.groupby(
    ["domain", "complexity"], observed=True
):
    group = group.copy()
    group["log10_tokens_centered"] = np.log10(group["prompt_tokens"]) - np.log10(
        group["prompt_tokens"].median()
    )
    formula = (
        f"score ~ C(model, Treatment(reference='{REFERENCE_MODEL}'))"
        " * log10_tokens_centered"
    )
    fit = smf.glm(
        formula,
        data=group,
        family=sm.families.Binomial(),
    ).fit()
    for model in COMPARISON_MODELS:
        model_term = f"C(model, Treatment(reference='{REFERENCE_MODEL}'))[T.{model}]"
        interaction = model_term + ":log10_tokens_centered"
        trend_rows.append(
            {
                "domain": domain,
                "complexity": complexity,
                "architecture": model,
                "median_tokens": group["prompt_tokens"].median(),
                "odds_ratio_at_median": np.exp(fit.params[model_term]),
                "architecture_at_median_p": fit.pvalues[model_term],
                "architecture_x_10x_length_log_odds": fit.params[interaction],
                "length_interaction_p": fit.pvalues[interaction],
            }
        )
trend_df = pd.DataFrame(trend_rows).sort_values(
    ["domain", "complexity", "architecture"]
)
display(
    trend_df.style.format(
        {
            "median_tokens": "{:,.0f}",
            "odds_ratio_at_median": "{:.2f}",
            "architecture_at_median_p": "{:.2g}",
            "architecture_x_10x_length_log_odds": "{:+.3f}",
            "length_interaction_p": "{:.2g}",
        }
    )
)